#Phase 1 — Load the Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/final_uci_drug_review_ADE_sentiment.csv")

print(df.shape)
df.head()

(161297, 15)


,uniqueID,drugName,condition,review,rating,date,usefulCount,ade_entities,ade_count,silver_label,max_confidence,max_severity,max_frequency,sentiment,sentiment_score
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,2012-05-20,27,[],0,0,0.000,0,0,Neutral,0.646
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,2010-04-27,192,['cranky'],1,1,0.806,0,0,Neutral,0.435
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,2009-12-14,17,['brown discharge'],1,1,0.920,0,5,Neutral,0.502
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,2015-11-03,10,[],0,0,0.000,0,0,Positive,0.847
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,2016-11-27,37,['constipation'],1,1,0.950,1,0,Positive,0.899


#Phase 2 — Inspect the Dataset

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161297 entries, 0 to 161296
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   uniqueID         161297 non-null  int64  
 1   drugName         161297 non-null  object 
 2   condition        159239 non-null  object 
 3   review           161297 non-null  object 
 4   rating           161297 non-null  int64  
 5   date             161297 non-null  object 
 6   usefulCount      161297 non-null  int64  
 7   ade_entities     161297 non-null  object 
 8   ade_count        161297 non-null  int64  
 9   silver_label     161297 non-null  int64  
 10  max_confidence   161297 non-null  float64
 11  max_severity     161297 non-null  int64  
 12  max_frequency    161297 non-null  int64  
 13  sentiment        161297 non-null  object 
 14  sentiment_score  161297 non-null  float64
dtypes: float64(2), int64(7), object(6)
memory usage: 18.5+ MB


In [ ]:
missing = pd.DataFrame({
    "Missing": df.isnull().sum(),
    "Percentage": round(df.isnull().mean()*100,2)
})

missing.sort_values(
    "Missing",
    ascending=False
)

,Missing,Percentage
condition,2058,1.28
uniqueID,0,0.00
drugName,0,0.00
review,0,0.00
rating,0,0.00
date,0,0.00
usefulCount,0,0.00
ade_entities,0,0.00
ade_count,0,0.00
silver_label,0,0.00


In [ ]:
df.dropna(inplace=True)

In [ ]:
print(df.shape)
print(df.isnull().sum())

(159239, 15)
uniqueID           0
drugName           0
condition          0
review             0
rating             0
date               0
usefulCount        0
ade_entities       0
ade_count          0
silver_label       0
max_confidence     0
max_severity       0
max_frequency      0
sentiment          0
sentiment_score    0
dtype: int64


In [ ]:
df["silver_label"].value_counts()

,count
silver_label,
1,110282
0,48957


In [ ]:
df.describe()

,uniqueID,rating,usefulCount,ade_count,silver_label,max_confidence,max_severity,max_frequency,sentiment_score
count,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000,159239.000000
mean,116032.886812,6.997406,28.186989,1.667858,0.692556,0.644849,0.619484,0.147803,0.701941
std,66961.789469,3.272545,36.521902,1.851003,0.461436,0.429560,1.463653,0.777924,0.177051
min,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.335000
25%,58237.000000,5.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.542000
50%,115893.000000,8.000000,16.000000,1.000000,1.000000,0.950000,0.000000,0.000000,0.707000
75%,173863.500000,10.000000,37.000000,2.000000,1.000000,0.950000,0.000000,0.000000,0.869000
max,232291.000000,10.000000,1291.000000,31.000000,1.000000,0.971000,5.000000,5.000000,0.991000


#Phase 3 — Feature Engineering

##Step 1 — Encode Sentiment
*Since sentiment is text, convert it to numbers.*

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])
print(dict(zip(
    label_encoder.classes_,
    label_encoder.transform(label_encoder.classes_)
)))

{'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}


##Step 2 — TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1,2),
    min_df=5
)

In [ ]:
X_text = tfidf.fit_transform(df["review"])

print(X_text.shape)

(159239, 5000)


##Step 3 — Define Feature Lists

###1. Hybrid features

In [ ]:
hybrid_features = [
    "rating",
    "max_severity",
    "max_frequency"
]

###2. Sentiment features

In [ ]:
sentiment_features = [
    "sentiment_encoded",
    "sentiment_score"
]

##Step 4 — Define Target

In [ ]:
y = df["silver_label"]

QUICK CHECK

In [ ]:
print("Dataset shape:", df.shape)
print("TF-IDF shape:", X_text.shape)

print("\nHybrid features:")
print(df[hybrid_features].head())

print("\nSentiment features:")
print(df[sentiment_features].head())

print("\nTarget distribution:")
print(y.value_counts())

Dataset shape: (159239, 16)
TF-IDF shape: (159239, 5000)

Hybrid features:
   rating  max_severity  max_frequency
0       9             0              0
1       8             0              0
2       5             0              5
3       8             0              0
4       9             1              0

Sentiment features:
   sentiment_encoded  sentiment_score
0                  1            0.646
1                  1            0.435
2                  1            0.502
3                  2            0.847
4                  2            0.899

Target distribution:
silver_label
1    110282
0     48957
Name: count, dtype: int64


#Phase 4 — Construct the Three Feature Sets

**Feature Set A (Baseline)**
1. TF-IDF (review)

**Feature Set B**
1. TF-IDF
2. Rating
3. Max Severity
4. Max Frequency

**Feature Set C**
1. TF-IDF
2. Rating
3. Max Severity
4. Max Frequency
5. Sentiment Encoded
6. Sentiment Score

In [ ]:
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

##Feature Set A (Baseline)

In [ ]:
X_A = X_text

##Feature Set B (Hybrid NLP Features)

In [ ]:
X_B = hstack([
    X_text,
    csr_matrix(df[hybrid_features].values)
])

##Feature Set C (Hybrid + Sentiment)

In [ ]:
all_features = hybrid_features + sentiment_features
X_C = hstack([
    X_text,
    csr_matrix(df[all_features].values)
])

SHAPE

In [ ]:
print("Feature Set A:", X_A.shape)
print("Feature Set B:", X_B.shape)
print("Feature Set C:", X_C.shape)

Feature Set A: (159239, 5000)
Feature Set B: (159239, 5003)
Feature Set C: (159239, 5005)


#Phase 5 — Train / Validation / Test Split

70% Train
10% Validation
20% Test

We'll do it in two steps using stratified sampling.

##Step 1 — Import

In [ ]:
from sklearn.model_selection import train_test_split

##Step 2 — Split off the Test Set (20%)

### 1. Feature Set A

In [ ]:
XA_trainval, XA_test, y_trainval, y_test = train_test_split(
    X_A,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

###2. Feature Set B

In [ ]:
XB_trainval, XB_test, _, _ = train_test_split(
    X_B,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

###3. Feature Set C

In [ ]:
XC_trainval, XC_test, _, _ = train_test_split(
    X_C,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

##Step 3 — Create Validation Set

Now split the remaining 80%.
We want :
Train = 70%,
Validation = 10%

Since validation should be 10 out of the remaining 80, we use - 10 / 80 = 0.125

### 1. Feature Set A

In [ ]:
XA_train, XA_val, y_train, y_val = train_test_split(
    XA_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_trainval
)

###2. Feature Set B

In [ ]:
XB_train, XB_val, _, _ = train_test_split(
    XB_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_trainval
)

###3. Feature Set C

In [ ]:
XC_train, XC_val, _, _ = train_test_split(
    XC_trainval,
    y_trainval,
    test_size=0.125,
    random_state=42,
    stratify=y_trainval
)

VERIFY

In [ ]:
print("TRAIN :", XA_train.shape)
print("VALID :", XA_val.shape)
print("TEST  :", XA_test.shape)

TRAIN : (111467, 5000)
VALID : (15924, 5000)
TEST  : (31848, 5000)


##Step 5 — Verify Label Distribution

In [ ]:
print("Train")
print(y_train.value_counts(normalize=True))

print("\nValidation")
print(y_val.value_counts(normalize=True))

print("\nTest")
print(y_test.value_counts(normalize=True))

Train
silver_label
1    0.692555
0    0.307445
Name: proportion, dtype: float64

Validation
silver_label
1    0.69254
0    0.30746
Name: proportion, dtype: float64

Test
silver_label
1    0.692571
0    0.307429
Name: proportion, dtype: float64


#Phase 6 — Random Forest Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

##Model Creation

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

#Phase 7 — Applying Train-Validation-Test on the Model

#Phase 7.1 — Feature Set A:

##Train

In [ ]:
rf_model.fit(
    XA_train,
    y_train
)

RandomForestClassifier(max_depth=15, n_estimators=200, n_jobs=-1,
                       random_state=42)

## Validation

In [ ]:
val_predictions = rf_model.predict(XA_val)
val_probabilities = rf_model.predict_proba(XA_val)[:,1]

print("Validation Accuracy :", accuracy_score(y_val, val_predictions))
print("Validation Precision:", precision_score(y_val, val_predictions))
print("Validation Recall   :", recall_score(y_val, val_predictions))
print("Validation F1 Score :", f1_score(y_val, val_predictions))
print("Validation ROC AUC  :", roc_auc_score(y_val, val_probabilities))

Validation Accuracy : 0.693983923637277
Validation Precision: 0.6935656330586829
Validation Recall   : 0.9999093217265144
Validation F1 Score : 0.8190292271697552
Validation ROC AUC  : 0.8440661145367347


## Test

In [ ]:
test_predictions = rf_model.predict(XA_test)
test_probabilities = rf_model.predict_proba(XA_test)[:,1]

In [ ]:
accuracy = accuracy_score(y_test, test_predictions)

precision = precision_score(y_test, test_predictions)
recall = recall_score(y_test, test_predictions)
f1 = f1_score(y_test, test_predictions)
roc_auc = roc_auc_score(y_test, test_probabilities)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

Accuracy : 0.6942
Precision: 0.6937
Recall   : 0.9999
F1 Score : 0.8191
ROC-AUC  : 0.8426


##Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    test_predictions
)
print(cm)

[[   54  9737]
 [    3 22054]]


##Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        test_predictions
    )
)

              precision    recall  f1-score   support

           0       0.95      0.01      0.01      9791
           1       0.69      1.00      0.82     22057

    accuracy                           0.69     31848
   macro avg       0.82      0.50      0.42     31848
weighted avg       0.77      0.69      0.57     31848



#Phase 7.2 — Feature Set B (TF-IDF + Hybrid Features)

## Train

In [ ]:
# ==========================================================
# Random Forest - Feature Set B
# ==========================================================

rf_model_B = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_model_B.fit(
    XB_train,
    y_train
)

RandomForestClassifier(max_depth=15, n_estimators=200, n_jobs=-1,
                       random_state=42)

## Validation

In [ ]:
# Validation Predictions
val_predictions_B = rf_model_B.predict(XB_val)
val_probabilities_B = rf_model_B.predict_proba(XB_val)[:, 1]

print("===== Feature Set B : Validation =====")

print(f"Accuracy : {accuracy_score(y_val, val_predictions_B):.4f}")
print(f"Precision: {precision_score(y_val, val_predictions_B):.4f}")
print(f"Recall   : {recall_score(y_val, val_predictions_B):.4f}")
print(f"F1 Score : {f1_score(y_val, val_predictions_B):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, val_probabilities_B):.4f}")

===== Feature Set B : Validation =====
Accuracy : 0.6947
Precision: 0.6941
Recall   : 0.9998
F1 Score : 0.8194
ROC-AUC  : 0.8651


##Test

In [ ]:
# Test Predictions

test_predictions_B = rf_model_B.predict(XB_test)

test_probabilities_B = rf_model_B.predict_proba(XB_test)[:, 1]

In [ ]:
accuracy_B = accuracy_score(y_test, test_predictions_B)
precision_B = precision_score(y_test, test_predictions_B)
recall_B = recall_score(y_test, test_predictions_B)
f1_B = f1_score(y_test, test_predictions_B)
roc_auc_B = roc_auc_score(y_test, test_probabilities_B)

print("===== Feature Set B : Test =====")

print(f"Accuracy : {accuracy_B:.4f}")
print(f"Precision: {precision_B:.4f}")
print(f"Recall   : {recall_B:.4f}")
print(f"F1 Score : {f1_B:.4f}")
print(f"ROC-AUC  : {roc_auc_B:.4f}")

===== Feature Set B : Test =====
Accuracy : 0.6945
Precision: 0.6940
Recall   : 0.9997
F1 Score : 0.8193
ROC-AUC  : 0.8643


#Phase 7.3 — Feature Set C (TF-IDF + Hybrid + Sentiment)

## Train

In [ ]:
# ==========================================================
# Random Forest - Feature Set C
# ==========================================================

rf_model_C = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_model_C.fit(
    XC_train,
    y_train
)

RandomForestClassifier(max_depth=15, n_estimators=200, n_jobs=-1,
                       random_state=42)

##Validation

In [ ]:
# Validation Predictions
val_predictions_C = rf_model_C.predict(XC_val)
val_probabilities_C = rf_model_C.predict_proba(XC_val)[:, 1]

print("===== Feature Set C : Validation =====")

print(f"Accuracy : {accuracy_score(y_val, val_predictions_C):.4f}")
print(f"Precision: {precision_score(y_val, val_predictions_C):.4f}")
print(f"Recall   : {recall_score(y_val, val_predictions_C):.4f}")
print(f"F1 Score : {f1_score(y_val, val_predictions_C):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, val_probabilities_C):.4f}")

===== Feature Set C : Validation =====
Accuracy : 0.7050
Precision: 0.7018
Recall   : 0.9982
F1 Score : 0.8242
ROC-AUC  : 0.8624


##Test

In [ ]:
# Test Predictions

test_predictions_C = rf_model_C.predict(XC_test)
test_probabilities_C = rf_model_C.predict_proba(XC_test)[:, 1]

accuracy_C = accuracy_score(y_test, test_predictions_C)
precision_C = precision_score(y_test, test_predictions_C)
recall_C = recall_score(y_test, test_predictions_C)
f1_C = f1_score(y_test, test_predictions_C)
roc_auc_C = roc_auc_score(y_test, test_probabilities_C)

print("===== Feature Set C : Test =====")

print(f"Accuracy : {accuracy_C:.4f}")
print(f"Precision: {precision_C:.4f}")
print(f"Recall   : {recall_C:.4f}")
print(f"F1 Score : {f1_C:.4f}")
print(f"ROC-AUC  : {roc_auc_C:.4f}")

===== Feature Set C : Test =====
Accuracy : 0.7047
Precision: 0.7015
Recall   : 0.9986
F1 Score : 0.8241
ROC-AUC  : 0.8619


##Final Comparison Table — Random Forest

In [ ]:
rf_results = pd.DataFrame({
    "Feature Set": [
        "A (TF-IDF)",
        "B (TF-IDF + Hybrid)",
        "C (TF-IDF + Hybrid + Sentiment)"
    ],
    "Accuracy": [
        accuracy,
        accuracy_B,
        accuracy_C
    ],
    "Precision": [
        precision,
        precision_B,
        precision_C
    ],
    "Recall": [
        recall,
        recall_B,
        recall_C
    ],
    "F1 Score": [
        f1,
        f1_B,
        f1_C
    ],
    "ROC-AUC": [
        roc_auc,
        roc_auc_B,
        roc_auc_C
    ]
})

rf_results

,Feature Set,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,A (TF-IDF),0.694172,0.693718,0.999864,0.819120,0.842574
1,B (TF-IDF + Hybrid),0.694518,0.693995,0.999728,0.819268,0.864263
2,C (TF-IDF + Hybrid + Sentiment),0.704691,0.701452,0.998640,0.824071,0.861920


#Phase 8 — LightGBM Training

In [ ]:
!pip install lightgbm -q

In [ ]:
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

##Model Creation

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    random_state=42,
    n_jobs=-1
)

#Phase 9 — Applying Train-Validation-Test on the Model

#Phase 9.1 — Feature Set A:

##Train

In [ ]:
lgbm_model.fit(
    XA_train,
    y_train
)

[LightGBM] [Info] Number of positive: 77197, number of negative: 34270
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 15.409988 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 599157
[LightGBM] [Info] Number of data points in the train set: 111467, number of used features: 5000
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.692555 -> initscore=0.812090
[LightGBM] [Info] Start training from score 0.812090
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

LGBMClassifier(colsample_bytree=0.8, max_depth=6, n_estimators=200, n_jobs=-1,
               objective='binary', random_state=42, subsample=0.8)

## Validation

In [ ]:
val_predictions_lgbm = lgbm_model.predict(XA_val)
val_probabilities_lgbm = lgbm_model.predict_proba(XA_val)[:,1]

print("Validation Accuracy :", accuracy_score(y_val, val_predictions_lgbm))
print("Validation Precision:", precision_score(y_val, val_predictions_lgbm))
print("Validation Recall   :", recall_score(y_val, val_predictions_lgbm))
print("Validation F1 Score :", f1_score(y_val, val_predictions_lgbm))
print("Validation ROC AUC  :", roc_auc_score(y_val, val_probabilities_lgbm))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Validation Accuracy : 0.8191409193669932
Validation Precision: 0.8543224908679771
Validation Recall   : 0.8907326804497643
Validation F1 Score : 0.872147740388884
Validation ROC AUC  : 0.8794539682560848


## Test

In [ ]:
test_predictions_lgbm = lgbm_model.predict(XA_test)
test_probabilities_lgbm = lgbm_model.predict_proba(XA_test)[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
accuracy_lgbm = accuracy_score(y_test, test_predictions_lgbm)

precision_lgbm = precision_score(y_test, test_predictions_lgbm)
recall_lgbm = recall_score(y_test, test_predictions_lgbm)
f1_lgbm = f1_score(y_test, test_predictions_lgbm)
roc_auc_lgbm = roc_auc_score(y_test, test_probabilities_lgbm)

print(f"Accuracy : {accuracy_lgbm:.4f}")
print(f"Precision: {precision_lgbm:.4f}")
print(f"Recall   : {recall_lgbm:.4f}")
print(f"F1 Score : {f1_lgbm:.4f}")
print(f"ROC-AUC  : {roc_auc_lgbm:.4f}")

Accuracy : 0.8162
Precision: 0.8534
Recall   : 0.8870
F1 Score : 0.8699
ROC-AUC  : 0.8783


##Confusion Matrix

In [ ]:
cm_lgbm = confusion_matrix(
    y_test,
    test_predictions_lgbm
)
print(cm_lgbm)

[[ 6430  3361]
 [ 2492 19565]]


##Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        test_predictions_lgbm
    )
)

              precision    recall  f1-score   support

           0       0.72      0.66      0.69      9791
           1       0.85      0.89      0.87     22057

    accuracy                           0.82     31848
   macro avg       0.79      0.77      0.78     31848
weighted avg       0.81      0.82      0.81     31848



#Phase 9.2 — Feature Set B (TF-IDF + Hybrid Features)

## Train

In [ ]:
# ==========================================================
# LightGBM - Feature Set B
# ==========================================================

lgbm_model_B = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    random_state=42,
    n_jobs=-1
)

lgbm_model_B.fit(
    XB_train,
    y_train
)

[LightGBM] [Info] Number of positive: 77197, number of negative: 34270
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 6.535998 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 599180
[LightGBM] [Info] Number of data points in the train set: 111467, number of used features: 5003
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.692555 -> initscore=0.812090
[LightGBM] [Info] Start training from score 0.812090
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

LGBMClassifier(colsample_bytree=0.8, max_depth=6, n_estimators=200, n_jobs=-1,
               objective='binary', random_state=42, subsample=0.8)

## Validation

In [ ]:
# Validation Predictions
val_predictions_lgbm_B = lgbm_model_B.predict(XB_val)
val_probabilities_lgbm_B = lgbm_model_B.predict_proba(XB_val)[:, 1]

print("===== Feature Set B : Validation =====")

print(f"Accuracy : {accuracy_score(y_val, val_predictions_lgbm_B):.4f}")
print(f"Precision: {precision_score(y_val, val_predictions_lgbm_B):.4f}")
print(f"Recall   : {recall_score(y_val, val_predictions_lgbm_B):.4f}")
print(f"F1 Score : {f1_score(y_val, val_predictions_lgbm_B):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, val_probabilities_lgbm_B):.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


===== Feature Set B : Validation =====
Accuracy : 0.8283
Precision: 0.8769
Recall   : 0.8750
F1 Score : 0.8759
ROC-AUC  : 0.9004


##Test

In [ ]:
# Test Predictions

test_predictions_lgbm_B = lgbm_model_B.predict(XB_test)

test_probabilities_lgbm_B = lgbm_model_B.predict_proba(XB_test)[:, 1]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
accuracy_lgbm_B = accuracy_score(y_test, test_predictions_lgbm_B)
precision_lgbm_B = precision_score(y_test, test_predictions_lgbm_B)
recall_lgbm_B = recall_score(y_test, test_predictions_lgbm_B)
f1_lgbm_B = f1_score(y_test, test_predictions_lgbm_B)
roc_auc_lgbm_B = roc_auc_score(y_test, test_probabilities_lgbm_B)

print("===== Feature Set B : Test =====")

print(f"Accuracy : {accuracy_lgbm_B:.4f}")
print(f"Precision: {precision_lgbm_B:.4f}")
print(f"Recall   : {recall_lgbm_B:.4f}")
print(f"F1 Score : {f1_lgbm_B:.4f}")
print(f"ROC-AUC  : {roc_auc_lgbm_B:.4f}")

===== Feature Set B : Test =====
Accuracy : 0.8280
Precision: 0.8765
Recall   : 0.8750
F1 Score : 0.8757
ROC-AUC  : 0.9008


#Phase 9.3 — Feature Set C (TF-IDF + Hybrid + Sentiment)

## Train

In [ ]:
# ==========================================================
# LightGBM - Feature Set C
# ==========================================================

lgbm_model_C = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    random_state=42,
    n_jobs=-1
)

lgbm_model_C.fit(
    XC_train,
    y_train
)

[LightGBM] [Info] Number of positive: 77197, number of negative: 34270
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 6.460095 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 599438
[LightGBM] [Info] Number of data points in the train set: 111467, number of used features: 5005
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.692555 -> initscore=0.812090
[LightGBM] [Info] Start training from score 0.812090
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

LGBMClassifier(colsample_bytree=0.8, max_depth=6, n_estimators=200, n_jobs=-1,
               objective='binary', random_state=42, subsample=0.8)

##Validation

In [ ]:
# Validation Predictions
val_predictions_lgbm_C = lgbm_model_C.predict(XC_val)
val_probabilities_lgbm_C = lgbm_model_C.predict_proba(XC_val)[:, 1]

print("===== Feature Set C : Validation =====")

print(f"Accuracy : {accuracy_score(y_val, val_predictions_lgbm_C):.4f}")
print(f"Precision: {precision_score(y_val, val_predictions_lgbm_C):.4f}")
print(f"Recall   : {recall_score(y_val, val_predictions_lgbm_C):.4f}")
print(f"F1 Score : {f1_score(y_val, val_predictions_lgbm_C):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, val_probabilities_lgbm_C):.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


===== Feature Set C : Validation =====
Accuracy : 0.8293
Precision: 0.8733
Recall   : 0.8813
F1 Score : 0.8773
ROC-AUC  : 0.9000


##Test

In [ ]:
# Test Predictions

test_predictions_lgbm_C = lgbm_model_C.predict(XC_test)
test_probabilities_lgbm_C = lgbm_model_C.predict_proba(XC_test)[:, 1]

accuracy_lgbm_C = accuracy_score(y_test, test_predictions_lgbm_C)
precision_lgbm_C = precision_score(y_test, test_predictions_lgbm_C)
recall_lgbm_C = recall_score(y_test, test_predictions_lgbm_C)
f1_lgbm_C = f1_score(y_test, test_predictions_lgbm_C)
roc_auc_lgbm_C = roc_auc_score(y_test, test_probabilities_lgbm_C)

print("===== Feature Set C : Test =====")

print(f"Accuracy : {accuracy_lgbm_C:.4f}")
print(f"Precision: {precision_lgbm_C:.4f}")
print(f"Recall   : {recall_lgbm_C:.4f}")
print(f"F1 Score : {f1_lgbm_C:.4f}")
print(f"ROC-AUC  : {roc_auc_lgbm_C:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


===== Feature Set C : Test =====
Accuracy : 0.8272
Precision: 0.8730
Recall   : 0.8783
F1 Score : 0.8756
ROC-AUC  : 0.9000


##Final Comparison Table — LightGBM

In [ ]:
lgbm_results = pd.DataFrame({
    "Feature Set": [
        "A (TF-IDF)",
        "B (TF-IDF + Hybrid)",
        "C (TF-IDF + Hybrid + Sentiment)"
    ],
    "Accuracy": [
        accuracy_lgbm,
        accuracy_lgbm_B,
        accuracy_lgbm_C
    ],
    "Precision": [
        precision_lgbm,
        precision_lgbm_B,
        precision_lgbm_C
    ],
    "Recall": [
        recall_lgbm,
        recall_lgbm_B,
        recall_lgbm_C
    ],
    "F1 Score": [
        f1_lgbm,
        f1_lgbm_B,
        f1_lgbm_C
    ],
    "ROC-AUC": [
        roc_auc_lgbm,
        roc_auc_lgbm_B,
        roc_auc_lgbm_C
    ]
})

lgbm_results

,Feature Set,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,A (TF-IDF),0.816221,0.853398,0.88702,0.869884,0.878312
1,B (TF-IDF + Hybrid),0.827995,0.876470,0.87496,0.875715,0.900812
2,C (TF-IDF + Hybrid + Sentiment),0.827242,0.873045,0.87827,0.875650,0.899953


#Phase 10 — Overall Model Comparison (Random Forest vs LightGBM)

In [ ]:
overall_results = pd.DataFrame({
    "Model": [
        "Random Forest", "Random Forest", "Random Forest",
        "LightGBM", "LightGBM", "LightGBM"
    ],
    "Feature Set": [
        "A (TF-IDF)", "B (TF-IDF + Hybrid)", "C (TF-IDF + Hybrid + Sentiment)",
        "A (TF-IDF)", "B (TF-IDF + Hybrid)", "C (TF-IDF + Hybrid + Sentiment)"
    ],
    "Accuracy": [
        accuracy, accuracy_B, accuracy_C,
        accuracy_lgbm, accuracy_lgbm_B, accuracy_lgbm_C
    ],
    "Precision": [
        precision, precision_B, precision_C,
        precision_lgbm, precision_lgbm_B, precision_lgbm_C
    ],
    "Recall": [
        recall, recall_B, recall_C,
        recall_lgbm, recall_lgbm_B, recall_lgbm_C
    ],
    "F1 Score": [
        f1, f1_B, f1_C,
        f1_lgbm, f1_lgbm_B, f1_lgbm_C
    ],
    "ROC-AUC": [
        roc_auc, roc_auc_B, roc_auc_C,
        roc_auc_lgbm, roc_auc_lgbm_B, roc_auc_lgbm_C
    ]
})

overall_results

,Model,Feature Set,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Random Forest,A (TF-IDF),0.694172,0.693718,0.999864,0.819120,0.842574
1,Random Forest,B (TF-IDF + Hybrid),0.694518,0.693995,0.999728,0.819268,0.864263
2,Random Forest,C (TF-IDF + Hybrid + Sentiment),0.704691,0.701452,0.998640,0.824071,0.861920
3,LightGBM,A (TF-IDF),0.816221,0.853398,0.887020,0.869884,0.878312
4,LightGBM,B (TF-IDF + Hybrid),0.827995,0.876470,0.874960,0.875715,0.900812
5,LightGBM,C (TF-IDF + Hybrid + Sentiment),0.827242,0.873045,0.878270,0.875650,0.899953


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=overall_results)

https://docs.google.com/spreadsheets/d/1nxapBZqPk7tOBfAwVWq-44IIkV-egY7SoSYOyhekXTc/edit#gid=0
